In [0]:
#import statements
from pyspark.sql.functions import col, current_date, to_date

In [0]:
#read in claims.csv
df_claims = spark.read.option('header','true').csv('/Volumes/insurance_db/bronze/insurance_files/claims.csv')

In [0]:

df_claims = df_claims \
    .withColumn('claim_id', col('claim_id').cast('int')) \
    .withColumn('policy_id', col('policy_id').cast('int')) \
    .withColumn('claim_amount', col('claim_amount').cast('double')) \
    .withColumn('claim_date', to_date(col('claim_date')))

In [0]:
bad = df_claims.filter(col('policy_id').isNull() |
                   (col('claim_amount') <= 0) |
                   (col('claim_date') > current_date()))

In [0]:
bad = bad.withColumn('claim_id', col('claim_id').cast('int'))

In [0]:
good = df_claims.subtract(bad)

In [0]:
bad.write.format('delta') \
    .mode('overwrite') \
    .option("overwriteSchema", "true") \
    .saveAsTable('insurance_db.quarantine.bad_claims')

In [0]:
good.write.format('delta') \
    .mode('overwrite') \
    .option("overwriteSchema", "true") \
    .saveAsTable('insurance_db.silver.claims')

In [0]:
print(f"Good: {good.count()} | Bad: {bad.count()}")

Good: 7 | Bad: 2


In [0]:
spark.sql('SELECT * FROM insurance_db.quarantine.bad_claims').show()


+--------+---------+------------+----------+--------+
|claim_id|policy_id|claim_amount|claim_date|  status|
+--------+---------+------------+----------+--------+
|     106|     NULL|       400.0|2024-07-01| pending|
|     107|        7|       250.0|2027-01-01|approved|
+--------+---------+------------+----------+--------+



In [0]:
spark.sql('SELECT * FROM insurance_db.silver.claims').show()

+--------+---------+------------+----------+--------+
|claim_id|policy_id|claim_amount|claim_date|  status|
+--------+---------+------------+----------+--------+
|     101|        1|       200.0|2024-02-01|approved|
|     105|        5|       700.0|2024-06-01|APPROVED|
|     109|        9|       350.0|2024-10-01| pending|
|     102|        2|       500.0|2024-03-01| pending|
|     104|        4|       150.0|2024-05-01|approved|
|     108|        8|       600.0|2024-09-01|  denied|
|     103|        3|       300.0|2024-04-01|  denied|
+--------+---------+------------+----------+--------+

